# TTD Price Assign — compute per-segment PoM from a delta policy

Given a universe of segments (e.g. `RankedByRevenue_March.csv`), a list of
**delta rules** (wildcard filters + a revenue policy + a delta), and an
optional **holdout list**, this notebook resolves one final price per segment
and writes it out in the format downstream systems expect.

**Output columns**: `Third_Party_Data_Provider_Id`, `Third_Party_Data_Provider_Element_Id`,
`Full_Path`, `PoM`, `delta`, `assign_date`.

## Design notes

- **Current price** comes from a second input file, `current_price.csv`
  (columns: `Third_Party_Data_Provider_Id`, `Third_Party_Data_Provider_Element_Id`,
  and a price column named `PoM`/`current_price`/`price`), joined onto the
  segment universe by provider + element id. A segment absent from that file
  falls back to `base_PoM_default[Third_Party_Data_Provider_Id]`. No prior
  notebook in this project has ever had a per-segment current-price source
  (`RankedByRevenue_March.csv` / `Price_Expt_Assignments2.csv` don't carry
  one), so a starter `current_price.csv` was generated reflecting the only
  price state actually known today: every segment at its provider's
  `base_PoM_default`. Replace it with real current-price data as soon as
  that's available — the join/fallback logic doesn't change either way.
- **Rule filter values can be a single wildcard OR a list of wildcards**,
  e.g. `'Third_Party_Data_Provider_Id': ['sharethis', 'dav2shreths']` matches
  either provider (OR, within that one rule) — on top of the existing OR
  across the whole `delta_rules` list (a segment gets a rule's delta if it
  matches *any* rule in the list). `revenue` accepts a list of policies the
  same way (e.g. `['<10', 'top:5']` keeps tiny segments AND the top 5).
- **Rule matching**: rules are evaluated independently (not in priority
  order) and a segment's final delta is the delta of whichever rule(s) match
  it. If a segment matches **more than one rule with different deltas**,
  this raises an error listing the conflicting segments/rules rather than
  silently picking one — narrow or reorder your rules' filters to resolve
  it. Matching the same delta from multiple rules is fine (not a conflict).
- **Unmatched segments**: a segment matching zero rules is treated the same
  as an explicit holdout — `delta=0`, still present in the output. Every
  segment in the input universe always gets exactly one output row.
- **Holdout always wins**: a segment in `holdout_segments` gets `delta=0`
  even if a rule would otherwise have matched it.

## Config — inputs you'll typically change

In [1]:
import pandas as pd
import numpy as np
import re

# ── current-price fallback (used where a segment is absent from current_price_file) ─
base_PoM_default = {'dav2shreths': 0.16, 'sharethis': 0.15}

# ── segment universe file ─────────────────────────────────────────────────────
data_path = "/Users/ravirajan/code/data"
output_path = "/Users/ravirajan/code/outputs/TTD"

segment_file = f"{data_path}/RankedByRevenueJuly.csv"   # e.g. RankedByRevenue.csv

# ── base price file (INPUT VARIABLE) ───────────────────────────────────────────
# Canonical, structural geo_rank/group reference (see base_price.csv design --
# rank/group are stable identifiers tied to the original experiment structure,
# not recomputed per round the way this round's own `revenue` is.
base_price_file = f"{output_path}/base_price.csv"

# ── reference file: the previous assignment (INPUT VARIABLE) ──────────────────
# Used both as this round's current-price source (it has a PoM column) AND to
# identify segments already changed by that prior assignment (see below).
# Points at 2026-05-26, not 2026-07-27 -- the July round was never pushed to
# production (abandoned in favor of this August round), so real segment
# prices/state are still whatever May 26 set them to.
reference_file = f"{output_path}/Price_Assign_2026-05-26.csv"

# ── manual holdout file (INPUT VARIABLE) ───────────────────────────────────────
# Full_Path values that must not change this round, on top of segments already
# changed by the previous assignment (computed below).
holdout_file = f"{output_path}/holdout_list_2026-08-06.csv"

# ── current price file (INPUT VARIABLE) ───────────────────────────────────────
# Columns: Third_Party_Data_Provider_Id, Third_Party_Data_Provider_Element_Id,
# and a price column named PoM / current_price / price.
current_price_file = reference_file

# ── assignment date (INPUT VARIABLE) ───────────────────────────────────────────
assign_date = "2026-08-06"

# ── campaign week-start this assignment applies to (INPUT VARIABLE) ────────────
week_start = "2026-08-10"


# ── output file (set to None to skip writing) ────────────────────────────────
output_file = f"{output_path}/Price_Assign_{assign_date}.csv"

In [2]:
def get_geo_from_path(x):
    """Same convention as TTD3_Pricing.ipynb / TTD_Pricing_Simple_Experiment.ipynb."""
    litems = x.replace(" ", "").split(">")
    check = ['Asia', 'US', 'Global', 'APAC']
    for c in check:
        if c in litems:
            return c
    return 'Unknown'

segments = pd.read_csv(segment_file)
# Missing revenue means "no measured revenue this period", not "unknown" --
# treated as $0/mo everywhere downstream (bucket assignment, the bulk -5pp
# rule's revenue<=1000 threshold, etc.), not silently dropped from rules that
# compare against revenue (NaN comparisons are always False in pandas, which
# would otherwise exclude these segments from a revenue-gated rule by
# accident rather than by an explicit decision).
segments['revenue'] = segments['revenue'].fillna(0)
segments['geo'] = segments['Full_Path'].apply(get_geo_from_path)

# Full_Path is the authoritative segment identity for every rule below (holdout,
# restore, bulk revenue threshold, experiment-2 membership all match on
# Full_Path, not Element_Id) -- but segment_file can carry more than one
# Third_Party_Data_Provider_Element_Id for the same (provider, Full_Path) (same
# taxonomy node, different underlying element). Collapse to one row per
# (provider, Full_Path): revenue sums (the node's true combined revenue), and
# Element_Id keeps ALL underlying ids joined with '|' (same convention as
# base_price_file) since they may still need individually addressing downstream.
segments = segments.groupby(['Third_Party_Data_Provider_Id', 'Full_Path'], as_index=False).agg(
    geo=('geo', 'first'),
    revenue=('revenue', lambda s: s.sum(min_count=1)),
    Third_Party_Data_Provider_Element_Id=('Third_Party_Data_Provider_Element_Id',
                                           lambda s: '|'.join(sorted(s.astype(str)))),
    _n_elements=('Third_Party_Data_Provider_Element_Id', 'size'),
)
n_collapsed = int((segments['_n_elements'] > 1).sum())
if n_collapsed:
    print(f"NOTE: {n_collapsed} Full_Path(s) collapsed from multiple Third_Party_Data_Provider_Element_Id "
          f"rows into one (revenue summed, Element_Id joined with '|')")
segments = segments.drop(columns='_n_elements')

# geo_rank/group are stable identifiers joined from base_price_file (not
# recomputed from this round's own revenue) -- see design notes above.
base_price_df = pd.read_csv(base_price_file)
segments = segments.merge(
    base_price_df[['Third_Party_Data_Provider_Id', 'Full_Path', 'geo_rank', 'group']],
    on=['Third_Party_Data_Provider_Id', 'Full_Path'], how='left'
)
n_missing_rank = segments['geo_rank'].isna().sum()
if n_missing_rank:
    print(f"NOTE: {n_missing_rank} segment(s) not found in base_price_file -- "
          f"geo_rank/group set to -1 (sentinel for 'not found') so the column stays plain int")
# sentinel -1 for segments absent from base_price_file (not a real rank/group) --
# avoids leaving blanks, so this reads back as plain int in any tool, not just
# pandas with nullable Int64 dtype requested.
segments['geo_rank'] = segments['geo_rank'].fillna(-1).astype(int)
segments['group'] = segments['group'].fillna(-1).astype(int)

print(f"{len(segments):,} segments loaded from {segment_file}")
print(segments.groupby(['Third_Party_Data_Provider_Id', 'geo']).size().unstack(fill_value=0))

reference_df = pd.read_csv(reference_file)
print(f"{len(reference_df):,} rows loaded from reference_file (prior assignment)")


NOTE: 11 Full_Path(s) collapsed from multiple Third_Party_Data_Provider_Element_Id rows into one (revenue summed, Element_Id joined with '|')
NOTE: 9 segment(s) not found in base_price_file -- geo_rank/group set to -1 (sentinel for 'not found') so the column stays plain int
5,734 segments loaded from /Users/ravirajan/code/data/RankedByRevenueJuly.csv
geo                           APAC  Asia  Global   US  Unknown
Third_Party_Data_Provider_Id                                  
dav2shreths                      0     2    2143    0        1
sharethis                       12  1218    1020  831      507
5,726 rows loaded from reference_file (prior assignment)


## Holdout list (INPUT VARIABLE)

Full_Path values (exact match) whose price must NOT change, regardless of
what any delta rule below says. Leave empty for no holdout.

In [3]:
# Manually-specified holdout, loaded from a file this round -- the ONLY
# segments frozen at their current price. Previously-changed segments are no
# longer auto-excluded here (unlike before): the bulk rule below now
# computes each eligible segment's delta as (target - PoM_old) rather than a
# flat -0.05, so a segment already at the target naturally resolves to
# delta=0 on its own -- no need to pre-exclude it to avoid double-treating
# it. This also means a segment sitting at the WRONG price (the historical
# +5pp test arm, or a former holdout member not reselected this round) gets
# correctly converged to the target instead of being permanently frozen
# there, which a flat "don't touch anything previously changed" rule would
# have done.
manual_holdout_df = pd.read_csv(holdout_file)
manual_holdout_segments = manual_holdout_df['Full_Path'].tolist()
print(f"{len(manual_holdout_segments):,} segment(s) in the manual holdout file ({holdout_file})")

holdout_segments = list(dict.fromkeys(manual_holdout_segments))
print(f"{len(holdout_segments):,} segment(s) held out in total")


230 segment(s) in the manual holdout file (/Users/ravirajan/code/outputs/TTD/holdout_list_2026-08-03.csv)
230 segment(s) held out in total


## Delta rules (INPUT VARIABLE)

A `delta_rules` entry is a dict:
```
{'Third_Party_Data_Provider_Id': <wildcard>, 'geo': <wildcard>, 'Full_Path': <wildcard>,
 'revenue': <policy>, 'delta': <float>}
```

- **Any column in `segments`** can be used as a filter key (not just the 3
  shown above) -- whatever key you give must match an actual column name, or
  the notebook raises a clear error rather than silently matching nothing.
- **Wildcards** use SQL `LIKE` syntax: `%` matches any sequence of characters,
  `_` matches exactly one character. `'%'` alone (or omitting a key entirely)
  matches everything for that column.
- **A filter value can be a single wildcard OR a list of wildcards** --
  `'Third_Party_Data_Provider_Id': ['sharethis', 'dav2shreths']` matches
  either provider (OR'd together within that one rule). This is on top of
  the OR that already exists across the whole `delta_rules` list itself --
  a segment gets a rule's delta if it matches *any* rule in the list, and
  within one rule's field it matches if it hits *any* pattern you give.
- **`revenue`** is a policy string (or a callable, see below), not a literal
  value -- it's evaluated *after* the other filters narrow the segment set,
  so `'top:50'` means "top 50 by revenue among segments already matching
  provider/geo/Full_Path", not top 50 of the whole universe.
- **`delta`** is required on every rule.

### Revenue policy syntax

| policy | meaning |
|---|---|
| `'*'`, `'all'`, or omitted | no revenue filter |
| `'>1000'`, `'>=1000'`, `'<100'`, `'<=100'`, `'==500'`, `'!=0'` | simple comparison |
| `'between:100,1000'` | inclusive range |
| `'top:50'` | top 50 rows by revenue (within the already-filtered subset) |
| `'bottom:50'` | bottom 50 rows by revenue (within the already-filtered subset) |
| `'toppct:10'` | top 10% by revenue, i.e. revenue >= the 90th percentile (within subset) |
| a callable `fn(sub_df) -> bool Series or filtered df` | full custom logic |
| a **list** of any of the above, e.g. `['<10', 'top:5']` | OR'd together -- keeps rows matching *any* listed policy |

### Sample usage

In [4]:
delta_rules = [
    # 1. Decrease price 5% for Global sharethis segments earning > $1000/month
    {'Third_Party_Data_Provider_Id': 'sharethis', 'geo': 'Global', 'Full_Path': '%',
     'revenue': '>1000', 'delta': -0.05},

    # 2. Increase price 5% for the top 50 dav2shreths segments by revenue, Global geo
    #    (scoped to a DIFFERENT provider than rule 1, so no overlap is possible)
    {'Third_Party_Data_Provider_Id': 'dav2shreths', 'geo': 'Global', 'Full_Path': '%',
     'revenue': 'top:50', 'delta': 0.05},

    # 3. Target a taxonomy branch via wildcard, no revenue filter at all
    #    (scoped to US geo -- disjoint from rules 1/2's Global geo, so no overlap)
    {'Third_Party_Data_Provider_Id': 'sharethis', 'geo': 'US',
     'Full_Path': 'US > shopping%', 'delta': -0.03},

    # 4. Any provider, segments in a specific monthly-revenue band
    #    (scoped to Asia geo -- disjoint from rules 1/2/3, so no overlap regardless
    #    of provider/Full_Path)
    {'geo': 'Asia', 'Full_Path': '%', 'revenue': 'between:100,1000', 'delta': -0.02},

    # 5. OR-list example: a filter value can be a list, OR'd together within
    #    that one field -- matches either provider AND either taxonomy branch
    #    (scoped to APAC geo -- disjoint from rules 1-4, so no overlap)
    {'Third_Party_Data_Provider_Id': ['sharethis', 'dav2shreths'], 'geo': 'APAC',
     'Full_Path': ['%Age%', '%Gender%'], 'delta': -0.04},

    # 6. Custom revenue policy via a lambda (equivalent to example 4's revenue
    #    condition, shown for reference if you need logic the string DSL can't express)
    # {'geo': 'Asia', 'Full_Path': '%', 'revenue': lambda sub: sub['revenue'].between(100, 1000), 'delta': -0.02},

    # 7. revenue OR-list example: keep tiny segments AND the top 5 by revenue
    # {'geo': 'US', 'Full_Path': '%', 'revenue': ['<10', 'top:5'], 'delta': -0.01},

    # 8. Catch-all: everything else not already covered above gets a small
    #    decrease. Put broad/catch-all rules LAST -- if this also matched
    #    segments rules 1-5 already touched, it would need to agree on the
    #    same delta for them or the notebook raises a conflict error.
    # {'Full_Path': '%', 'revenue': '*', 'delta': -0.01},
]

### Reconstruct real experiment deltas (optional override)

To generate actual price changes matching a *known* per-segment assignment
(e.g. `Price_Expt_Assignments2.csv`, the real pricing experiment analyzed in
`TTD3_Pricing.ipynb`) instead of the wildcard-based sample rules above, set
`use_real_experiment_deltas = True` below. This builds one **exact-match**
rule per nonzero-delta segment in the experiment file -- segments in that
file with `delta==0` (controls), or segments not in the experiment at all,
don't need a rule; they get `delta=0` from the notebook's normal
unmatched-segment default, same as everywhere else.

In [5]:
use_real_experiment_deltas = False  # this run uses the bulk+restore rules below instead
experiment_file = f"{data_path}/Price_Expt_Assignments2.csv"

def build_delta_rules_from_experiment(experiment_file):
    exp = pd.read_csv(experiment_file)
    exp['delta'] = exp['assignment'] * exp['incr']
    nz = exp[exp['delta'] != 0]
    dupes = nz[nz.duplicated(subset=['Third_Party_Data_Provider_Id', 'Full_Path'], keep=False)]
    if len(dupes) > 0:
        raise ValueError(
            f"{experiment_file} has {len(dupes)} row(s) with duplicate "
            f"(Third_Party_Data_Provider_Id, Full_Path) -- can't build exact-match "
            f"rules unambiguously from it:\n{dupes[['Third_Party_Data_Provider_Id', 'Full_Path', 'delta']].to_string(index=False)}"
        )
    return [
        {'Third_Party_Data_Provider_Id': row['Third_Party_Data_Provider_Id'],
         'Full_Path': row['Full_Path'], 'delta': row['delta']}
        for _, row in nz.iterrows()
    ]

if use_real_experiment_deltas:
    delta_rules = build_delta_rules_from_experiment(experiment_file)
    print(f"delta_rules overridden: {len(delta_rules):,} exact-match rules reconstructed from {experiment_file}")

import os

JOIN_KEYS = ['Third_Party_Data_Provider_Id', 'Full_Path']

def load_current_price(current_price_file):
    if not os.path.exists(current_price_file):
        raise FileNotFoundError(
            f"current_price_file not found: {current_price_file!r}\n"
            f"Expected a CSV with columns {JOIN_KEYS + ['PoM']} "
            f"(price column name can also be 'current_price' or 'price')."
        )
    cp = pd.read_csv(current_price_file)
    price_col = next((c for c in ['PoM', 'current_price', 'price'] if c in cp.columns), None)
    if price_col is None:
        raise KeyError(
            f"{current_price_file} has no recognized price column -- expected one of "
            f"'PoM' / 'current_price' / 'price', got columns {cp.columns.tolist()}"
        )
    missing_keys = [k for k in JOIN_KEYS if k not in cp.columns]
    if missing_keys:
        raise KeyError(f"{current_price_file} is missing join key column(s): {missing_keys}")

    # Full_Path is the authoritative segment identity for this lookup -- a
    # Full_Path can span multiple Third_Party_Data_Provider_Element_Id rows in
    # the source file, so average their price rather than treating it as a
    # data error (matches the same collapse convention used for `segments`).
    dupes = cp[cp.duplicated(subset=JOIN_KEYS, keep=False)]
    if len(dupes) > 0:
        n_paths = dupes.groupby(JOIN_KEYS).ngroups
        print(f"NOTE: {n_paths} Full_Path(s) in {current_price_file} span multiple "
              f"Element_Ids -- averaging their {price_col} for the current-price lookup")
    cp = cp.groupby(JOIN_KEYS, as_index=False)[price_col].mean()
    return cp.rename(columns={price_col: '_current_price'})


def get_current_price(df, current_price_file, base_PoM_default):
    """Per-segment current PoM: join current_price_file by
    (Third_Party_Data_Provider_Id, Full_Path), falling back to
    base_PoM_default[provider] for any segment absent from that file -- see
    design notes above."""
    provider_default = df['Third_Party_Data_Provider_Id'].map(base_PoM_default)
    if provider_default.isna().any():
        missing = df.loc[provider_default.isna(), 'Third_Party_Data_Provider_Id'].unique()
        raise KeyError(f"base_PoM_default has no entry for provider(s): {list(missing)}")

    cp = load_current_price(current_price_file)
    merged = df.merge(cp, on=JOIN_KEYS, how='left')
    assert len(merged) == len(df), "current_price join changed row count -- check for duplicate keys"
    n_from_file = int(merged['_current_price'].notna().sum())
    print(f"current price: {n_from_file:,} segment(s) from {current_price_file}, "
          f"{len(df) - n_from_file:,} fell back to base_PoM_default")
    return merged['_current_price'].where(merged['_current_price'].notna(), provider_default.values)

### This run's actual policy: bulk decrease + restore list

Two pieces, combined into `delta_rules`:

1. **Bulk rule**: `-0.05` for all `sharethis` segments with `revenue <= 1000`,
   excluding segments already changed by the previous assignment
   (`holdout_segments`, computed above) and excluding `restore_segments`
   (below) — computed as an explicit `Full_Path` list (not a wildcard +
   revenue-policy rule) so there's no possibility of it overlapping with
   the restore rules or holdout by construction, rather than relying on
   rule-priority (this engine has none -- see design notes at the top).
2. **Restore rules**: for `restore_segments`, bring the price back to
   `base_PoM_default[provider]` — computed *per segment* from its own
   current price in `reference_df` (not a fixed delta, since different
   segments moved by different amounts in the prior assignment).

In [6]:
restore_segments = [
    'Global > Validated Demographic > Age > 18-44',
    'Global > people & society > social issues & advocacy > green living & environmental issues',
    'Global > hobbies & leisure > outdoors',
    'Global > travel > tourist destinations > mountain & ski resorts',
    'Global > business & industrial > hospitality industry > food service',
    'Global > law & government > military > veterans',
    'Global > computers & electronics > networking > vpn & remote access',
    'Global > finance > investing',
    'Global > Validated Demographic > Gender and Age > Females 18-34',  # corrected from "Females" -- no exact match otherwise
    'Global > business & industrial > transportation & logistics > freight & trucking',
]

# restore_segments always wins over the "already changed" holdout -- remove
# any overlap so the restore rule isn't silently zeroed out by holdout.
_overlap = set(restore_segments) & set(holdout_segments)
if _overlap:
    print(f"{len(_overlap)} restore segment(s) removed from holdout_segments so their restore rule applies:")
    for s in sorted(_overlap):
        print(" ", s)
    holdout_segments = [s for s in holdout_segments if s not in restore_segments]


def build_restore_rules(restore_segments, reference_df, base_PoM_default, price_col='PoM'):
    """One exact-match rule per restore segment, delta computed to bring its
    CURRENT price (from reference_df) back to its provider's default."""
    sub = reference_df[reference_df['Full_Path'].isin(restore_segments)].copy()
    missing = set(restore_segments) - set(sub['Full_Path'])
    if missing:
        print(f"WARNING: {len(missing)} restore segment(s) not found in reference_df: {sorted(missing)}")
    sub['_target'] = sub['Third_Party_Data_Provider_Id'].map(base_PoM_default)
    sub['_restore_delta'] = (sub['_target'] - sub[price_col]).round(6)
    return [
        {'Third_Party_Data_Provider_Id': row['Third_Party_Data_Provider_Id'],
         'Full_Path': row['Full_Path'], 'delta': row['_restore_delta']}
        for _, row in sub.iterrows()
    ]

restore_rules = build_restore_rules(restore_segments, reference_df, base_PoM_default)
print(f"\n{len(restore_rules)} restore rule(s):")
for r in restore_rules:
    print(" ", r)

# current_price is needed here (not just later, where it fed `output['PoM_old']`)
# to compute each bulk-target segment's own convergence delta below.
current_price = get_current_price(segments, current_price_file, base_PoM_default)
_current_price_by_key = dict(zip(zip(segments['Third_Party_Data_Provider_Id'], segments['Full_Path']),
                                  current_price))

# T1's revenue<=1000 gate is evaluated against MAY revenue (reference_df's
# own 'revenue' column -- whatever was current when THAT round ran), not
# `segments['revenue']` (this round's July refresh). The rollout rule is
# sticky by design: a segment's May-time revenue is what actually earned it
# this round's T1 membership (and hence its current $0.10 price); gating on
# July revenue instead would incorrectly re-litigate that decision and drop
# segments that have simply grown past $1,000 since (4 such segments exist
# this round -- still part of the rollout population, just no longer
# eligible for a fresh Group 2 perturbation on top of it, per
# TTD_stratified_selection.ipynb's separate 1000+ carve-out). A segment
# absent from reference_df (new since May) has no May revenue to gate on --
# falls back to its own (July) revenue instead.
_may_revenue_by_key = dict(zip(zip(reference_df['Third_Party_Data_Provider_Id'], reference_df['Full_Path']),
                                reference_df['revenue']))
segments['may_revenue'] = [_may_revenue_by_key.get((p, f), np.nan)
                            for p, f in zip(segments['Third_Party_Data_Provider_Id'], segments['Full_Path'])]
_n_new_since_may = int(segments['may_revenue'].isna().sum())
if _n_new_since_may:
    print(f"\nNOTE: {_n_new_since_may} segment(s) not in reference_df (new since May) -- "
          f"T1 gate falls back to this round's own revenue for them")
segments['may_revenue'] = segments['may_revenue'].fillna(segments['revenue'])

_excluded = set(holdout_segments) | set(restore_segments)
_bulk_target = segments[
    (segments['Third_Party_Data_Provider_Id'] == 'sharethis') &
    (segments['may_revenue'] <= 1000) &
    (~segments['Full_Path'].isin(_excluded))
]['Full_Path'].tolist()

def build_bulk_convergence_rules(bulk_target, base_PoM_default, current_price_by_key, provider='sharethis'):
    """One exact-match rule per bulk-target segment, delta computed to bring
    its CURRENT price to the single accepted -5pp target -- same
    target-convergence pattern as build_restore_rules above, generalized to
    the whole bulk population instead of just the special restore_segments
    correction. This handles a segment starting from ANY prior price
    uniformly: already at target -> delta 0 (a no-op, same as before);
    never touched (at base) -> the familiar -5pp; sitting at the historical
    +5pp test arm's price, or a released holdout price -- gets corrected to
    the SAME target instead of being left wherever a flat rule left it."""
    target = base_PoM_default[provider] - 0.05   # accepted price = base - 5pp
    rules = []
    for full_path in bulk_target:
        pom_old = current_price_by_key[(provider, full_path)]
        delta = round(target - pom_old, 6)
        rules.append({'Third_Party_Data_Provider_Id': provider, 'Full_Path': full_path, 'delta': delta})
    return rules, target

bulk_rules, _bulk_target_price = build_bulk_convergence_rules(_bulk_target, base_PoM_default, _current_price_by_key)
print(f"\nbulk convergence target: {len(_bulk_target):,} segments -> ${_bulk_target_price:.3f}")

delta_rules = bulk_rules + restore_rules
print(f"delta_rules: {len(delta_rules)} rule(s) total ({len(bulk_rules):,}-segment bulk convergence + {len(restore_rules)} restore rules)")


10 restore rule(s):
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > Validated Demographic > Age > 18-44', 'delta': 0.05}
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > Validated Demographic > Gender and Age > Females 18-34', 'delta': 0.05}
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > business & industrial > hospitality industry > food service', 'delta': 0.05}
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > business & industrial > transportation & logistics > freight & trucking', 'delta': -0.05}
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > computers & electronics > networking > vpn & remote access', 'delta': -0.05}
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > finance > investing', 'delta': 0.05}
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > hobbies & leisure > outdoors', 'delta': -0.05}
  {'Third_Party_Data_Provider

## Engine — wildcard matching, revenue policy, rule application

In [7]:
def like_to_regex(pattern):
    """SQL LIKE-style wildcard: % = any sequence, _ = any single char.
    Splits on the wildcard chars first and escapes only the literal chunks --
    escaping the whole string first (then trying to un-escape %/_) doesn't
    work, since re.escape doesn't actually escape % or _ in the first place."""
    pattern = str(pattern)
    parts = re.split(r'([%_])', pattern)
    regex_parts = []
    for part in parts:
        if part == '%':
            regex_parts.append('.*')
        elif part == '_':
            regex_parts.append('.')
        else:
            regex_parts.append(re.escape(part))
    return re.compile('^' + ''.join(regex_parts) + '$')


def _has_wildcard_chars(p):
    return isinstance(p, str) and ('%' in p or '_' in p)


def wildcard_match(series, pattern):
    """`pattern` can be a single wildcard string, or a list/tuple/set of
    wildcard strings -- matches if ANY of them match (OR). A pattern with no
    %/_ at all skips the regex entirely (vectorized equality) -- exact-match
    rules (e.g. reconstructing a known per-segment assignment) are common
    and this is meaningfully faster at scale."""
    patterns = pattern if isinstance(pattern, (list, tuple, set)) else [pattern]
    str_series = None
    mask = pd.Series(False, index=series.index)
    for p in patterns:
        if p is None or p == '%':
            return pd.Series(True, index=series.index)
        if str_series is None:
            str_series = series.astype(str)
        if _has_wildcard_chars(p):
            rx = like_to_regex(p)
            mask |= str_series.apply(lambda x: bool(rx.match(x)))
        else:
            mask |= (str_series == str(p))
    return mask


def _apply_single_revenue_policy(sub_df, policy):
    if policy is None or policy in ('*', 'all'):
        return sub_df
    if callable(policy):
        result = policy(sub_df)
        if isinstance(result, pd.Series) and result.dtype == bool:
            return sub_df[result]
        return result  # assume already filtered

    policy = policy.strip()
    m = re.match(r'^(>=|<=|==|!=|>|<)\s*(-?\d+\.?\d*)$', policy)
    if m:
        op, val = m.group(1), float(m.group(2))
        rev = sub_df['revenue']
        ops = {'>': rev.gt, '>=': rev.ge, '<': rev.lt, '<=': rev.le, '==': rev.eq, '!=': rev.ne}
        return sub_df[ops[op](val)]

    m = re.match(r'^between:\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)$', policy)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
        return sub_df[sub_df['revenue'].between(lo, hi)]

    m = re.match(r'^top:\s*(\d+)$', policy)
    if m:
        return sub_df.nlargest(int(m.group(1)), 'revenue')

    m = re.match(r'^bottom:\s*(\d+)$', policy)
    if m:
        return sub_df.nsmallest(int(m.group(1)), 'revenue')

    m = re.match(r'^toppct:\s*(-?\d+\.?\d*)$', policy)
    if m:
        pct = float(m.group(1))
        threshold = sub_df['revenue'].quantile(1 - pct / 100)
        return sub_df[sub_df['revenue'] >= threshold]

    raise ValueError(f"Unrecognized revenue policy: {policy!r}")


def apply_revenue_policy(sub_df, policy):
    """See the revenue-policy table in the markdown above. `policy` can also
    be a list of policies, OR'd together (union of rows matching any of
    them) -- e.g. ['<10', 'top:5'] keeps tiny segments AND the top 5."""
    if isinstance(policy, (list, tuple)):
        matched_idx = pd.Index([])
        for p in policy:
            matched_idx = matched_idx.union(_apply_single_revenue_policy(sub_df, p).index)
        return sub_df.loc[sub_df.index.intersection(matched_idx)]
    return _apply_single_revenue_policy(sub_df, policy)


FILTERABLE_META_KEYS = {'delta', 'revenue'}  # not matched via wildcard directly

def apply_rule(df, rule):
    sub = df
    for key, pattern in rule.items():
        if key in FILTERABLE_META_KEYS:
            continue
        if key not in df.columns:
            raise KeyError(f"Rule references unknown column {key!r}: {rule}")
        sub = sub[wildcard_match(sub[key], pattern)]
    if 'revenue' in rule:
        sub = apply_revenue_policy(sub, rule['revenue'])
    return sub

In [8]:
def assign_deltas(df, delta_rules, holdout_segments):
    """Resolve one delta per segment. Matching the same delta from multiple
    rules is fine; matching DIFFERENT deltas from multiple rules raises --
    see design notes above. Segments matching no rule (and not in the
    holdout) default to delta=0, same as an explicit holdout."""
    matches = {idx: [] for idx in df.index}
    for rule_idx, rule in enumerate(delta_rules):
        if 'delta' not in rule:
            raise KeyError(f"Rule {rule_idx} missing required 'delta' key: {rule}")
        matched = apply_rule(df, rule)
        for idx in matched.index:
            matches[idx].append((rule_idx, rule['delta']))

    resolved_delta, conflicts = {}, []
    for idx, m in matches.items():
        deltas = {d for _, d in m}
        if len(deltas) > 1:
            conflicts.append({'Full_Path': df.loc[idx, 'Full_Path'], 'matches': m})
        elif len(deltas) == 1:
            resolved_delta[idx] = deltas.pop()
        else:
            resolved_delta[idx] = 0.0

    if conflicts:
        msg = "\n".join(f"  {c['Full_Path']!r}: matched rules {c['matches']}" for c in conflicts[:20])
        more = f"\n  ... and {len(conflicts) - 20} more" if len(conflicts) > 20 else ""
        raise ValueError(
            f"{len(conflicts)} segment(s) matched multiple rules with DIFFERENT deltas -- "
            f"narrow or reorder your rules' filters to resolve:\n{msg}{more}"
        )

    delta_series = pd.Series(resolved_delta).reindex(df.index).fillna(0.0)
    holdout_mask = df['Full_Path'].isin(holdout_segments)
    n_held = int(holdout_mask.sum())
    delta_series = delta_series.where(~holdout_mask, 0.0)
    print(f"{n_held} segment(s) held out (delta forced to 0 regardless of rule matches)")
    return delta_series



## Run — resolve deltas, compute PoM, build output

In [9]:
delta_series = assign_deltas(segments, delta_rules, holdout_segments)
# current_price was already computed above (cell 11) -- needed there to build
# the bulk convergence rules' per-segment deltas.

output = segments[['Third_Party_Data_Provider_Id', 'Third_Party_Data_Provider_Element_Id',
                    'Full_Path', 'geo', 'revenue', 'geo_rank', 'group']].copy()
output['delta'] = delta_series
# experiment_delta is specifically the experiment-2 perturbation for this
# round -- 0 for every row here (none of this round's own rule resolution is
# an experiment-2 action), populated only for actual experiment-2 candidates
# by the overlay below. `delta` (not experiment_delta) is what carries the
# real outcome -- bulk/holdout/restore/retained deltas -- for every row.
output['experiment_delta'] = 0.0
output['PoM_old'] = current_price
output['PoM'] = current_price + output['delta']
output['assign_date'] = assign_date
output['week_start'] = week_start

# previous_state: this segment's `state` as of reference_df (the prior
# round) -- "O" (origin/base price) for any segment absent from reference_df
# entirely, since a brand-new segment is assumed to start at base PoM (same
# convention as the geo_rank/group -1 sentinel above).
prev_state_by_key = dict(zip(zip(reference_df['Third_Party_Data_Provider_Id'], reference_df['Full_Path']),
                              reference_df['state']))
output['previous_state'] = [prev_state_by_key.get((p, f), 'O')
                             for p, f in zip(output['Third_Party_Data_Provider_Id'], output['Full_Path'])]

# state: the state machine's value for this round. Defaults to previous_state
# (no transition) -- one default covers "no change", "retained", manual
# holdout, AND a pure rollback alike, since all four are the same outcome
# from the state machine's point of view (this round did not move the
# segment into a new experiment membership, whatever its price did). Only
# the bulk-rule adoption below is a real transition here; the experiment-2
# overlay further down can override again.
output['state'] = output['previous_state']
output.loc[output['Full_Path'].isin(_bulk_target), 'state'] = 'T1 -0.05'
# manual_holdout_segments: segments this round's holdout file explicitly
# designates as the Experiment 1 comparison group get labeled 'C1' -- same
# convention as previously_changed_segments that already carried a 'C1'
# label from an earlier round. Without this, a segment holding no other
# rule membership just inherits previous_state unchanged (e.g. still 'T1'
# from a much earlier round it was actually treated in), which describes
# its realized price correctly but not its role in THIS round's experiment
# design. Applied before the restore_segments override below, since a
# restore always wins over a plain holdout label if a segment is both.
output.loc[output['Full_Path'].isin(manual_holdout_segments), 'state'] = 'C1'
# restore_segments (see below) revert all the way to "O" -- these were
# mistakenly placed into experiment 1 and are being corrected as if that
# never happened, as opposed to "no change"/"retained"/holdout, which
# genuinely didn't move. The experiment-2 overlay further down can still
# override this to T2/C2 for the segments that are ALSO real exp2 candidates.
output.loc[output['Full_Path'].isin(restore_segments), 'state'] = 'O'

output = output[['Third_Party_Data_Provider_Id', 'Third_Party_Data_Provider_Element_Id',
                  'Full_Path', 'geo', 'revenue', 'geo_rank', 'group', 'PoM_old', 'PoM', 'delta',
                  'experiment_delta', 'assign_date', 'week_start', 'state', 'previous_state']]

print(f"\n{len(output):,} segments assigned, {(output['delta'] != 0).sum():,} with a nonzero delta")
print(output['delta'].value_counts().sort_index())
print()
print(output['state'].value_counts())
output.head(10)


230 segment(s) held out (delta forced to 0 regardless of rule matches)

5,734 segments assigned, 3,038 with a nonzero delta
delta
-0.10     314
-0.05    2719
 0.00    2696
 0.05       5
Name: count, dtype: int64

state
T1 -0.05    3342
O           2162
C1           230
Name: count, dtype: int64


,Third_Party_Data_Provider_Id,Third_Party_Data_Provider_Element_Id,Full_Path,geo,revenue,geo_rank,group,PoM_old,PoM,delta,experiment_delta,assign_date,week_start,state,previous_state
0,dav2shreths,0b2ac7e9,Data Alliance > Arts & Entertainment > Interes...,Global,0.04,1184,394,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
1,dav2shreths,705e46b8,Data Alliance > Arts & Entertainment > Interes...,Global,1.31,743,247,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
2,dav2shreths,ee5edb85,Data Alliance > Arts & Entertainment > Interes...,Global,2.14,161,53,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
3,dav2shreths,c7a6cf98,Data Alliance > Arts & Entertainment > Interes...,Global,55.71,52,17,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
4,dav2shreths,8d76185a,Data Alliance > Arts & Entertainment > Interes...,Global,0.00,1509,502,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
5,dav2shreths,949e18f4,Data Alliance > Arts & Entertainment > Interes...,Global,0.00,537,178,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
6,dav2shreths,ef8fddbb,Data Alliance > Arts & Entertainment > Interes...,Global,5.73,204,67,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
7,dav2shreths,8f228ba2,Data Alliance > Arts & Entertainment > Interes...,Global,0.00,1185,394,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
8,dav2shreths,af0db075,Data Alliance > Arts & Entertainment > Interes...,Global,0.13,321,106,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O
9,dav2shreths,97279965,Data Alliance > Arts & Entertainment > Interes...,Global,50.11,22,7,0.16,0.16,0.0,0.0,2026-08-03,2026-08-03,O,O


## Experiment 2 overlay (optional)

Applies AFTER the main resolution above, as a targeted override for a
specific candidate list (`experiment2_candidates_file`) — first restores
each candidate to its provider's `base_PoM_default` (same computed-delta
approach as `restore_segments` above, based on its own `PoM_old` from this
run), then adds an independent random `+/- experiment2_delta_magnitude` on
top. Wins over every other rule/holdout for these specific segments, same
precedence idea as `restore_segments`.

`tag` distinguishes segments that were ever allocated a `group` in the
original experiment (`Price_Expt_Assignments2.csv`, regardless of what their
delta was there — even a `delta=0` control-arm member counts) from ones that
were never part of it at all.

In [10]:
run_experiment2 = True
experiment2_candidates_file = f"{output_path}/experiment2_candidates_2026-08-06.csv"

if run_experiment2:
    exp2_df = pd.read_csv(experiment2_candidates_file)
    exp2_base_tag = exp2_df['tag'].str.replace(r' \(delta = .*\)$', '', regex=True)
    # translate the file's own action words into this round's state codes
    _STATE_MAP = {'experiment 2': 'T2', 'control 2': 'C2'}
    exp2_state_raw = exp2_base_tag.map(_STATE_MAP)
    if exp2_state_raw.isna().any():
        unknown = exp2_base_tag[exp2_state_raw.isna()].unique()
        raise ValueError(f"{experiment2_candidates_file} has unrecognized tag value(s): {list(unknown)}")

    # T2 gets a direction suffix from its own new_delta sign -- same reason as
    # T1's split (a bare "T2" can't tell you which arm without separately
    # checking price/delta). C2 has no direction: control means
    # new_delta==0 by definition, so there's nothing to disambiguate.
    def _resolve_exp2_state(state, new_delta):
        if state != 'T2':
            return state
        if new_delta > 0:
            return 'T2 +0.025'
        if new_delta < 0:
            return 'T2 -0.025'
        raise ValueError("T2 row with new_delta==0 -- should have been tagged control 2")

    exp2_state_raw = pd.Series(
        [_resolve_exp2_state(s, d) for s, d in zip(exp2_state_raw, exp2_df['new_delta'])],
        index=exp2_df.index
    )

    exp2_keys_set = set(zip(exp2_df['Third_Party_Data_Provider_Id'], exp2_df['Full_Path']))
    exp2_states = dict(zip(zip(exp2_df['Third_Party_Data_Provider_Id'], exp2_df['Full_Path']), exp2_state_raw))
    exp2_perturbation = dict(zip(zip(exp2_df['Third_Party_Data_Provider_Id'], exp2_df['Full_Path']), exp2_df['new_delta']))

    output_keys = list(zip(output['Third_Party_Data_Provider_Id'], output['Full_Path']))
    exp2_mask = pd.Series([k in exp2_keys_set for k in output_keys], index=output.index)
    n_found = int(exp2_mask.sum())
    if n_found < len(exp2_df):
        print(f"WARNING: only {n_found} / {len(exp2_df)} experiment2 candidates found in output")
    else:
        print(f"{n_found} / {len(exp2_df)} experiment2 candidates found in output")

    # Every exp2 candidate's perturbation applies on top of its CONVERGED
    # target price, not its raw pre-round PoM_old -- Group 2 was restricted
    # upstream to a clean single-price base specifically so this holds, but
    # that restriction only constrains WHICH segments get selected; it
    # doesn't change what PoM_old happens to record for a segment that
    # hasn't been rolled out yet (still 0.15, or a stale 0.10/0.20 from the
    # historical +5pp arm). Two convergence targets apply, mutually
    # exclusive by construction (restore_segments are >$1000 revenue,
    # _bulk_target is <=$1000, so a segment is never both):
    #   - restore_segments -> base_PoM_default (unchanged from before)
    #   - _bulk_target (the accepted rollout population) -> _bulk_target_price
    # A segment matching neither (shouldn't happen now that Group 2 only
    # draws from _bulk_target, but kept for safety) gets convergence_delta=0,
    # i.e. perturbation still applies on top of its raw PoM_old as before.
    is_restore = output['Full_Path'].isin(restore_segments)
    is_bulk_target = output['Full_Path'].isin(_bulk_target)
    base_default_series = output['Third_Party_Data_Provider_Id'].map(base_PoM_default)
    restore_delta = (base_default_series - output['PoM_old']).where(is_restore, 0.0)
    bulk_convergence_delta = (_bulk_target_price - output['PoM_old']).where(is_bulk_target, 0.0)
    convergence_delta = restore_delta + bulk_convergence_delta
    perturbation = pd.Series([exp2_perturbation.get(k, 0.0) for k in output_keys], index=output.index)
    new_delta = convergence_delta + perturbation
    new_pom = output['PoM_old'] + new_delta

    # Experiment-2 candidacy always sets state to T2/C2 -- this is this
    # round's real, active experiment membership, taking precedence over
    # whatever cell above set (including a restore_segments row that's also
    # an exp2 candidate: rolling back and re-entering a new experiment are
    # not mutually exclusive).
    exp2_state_series = pd.Series([exp2_states.get(k) for k in output_keys], index=output.index)

    output.loc[exp2_mask, 'delta'] = new_delta[exp2_mask]
    output.loc[exp2_mask, 'experiment_delta'] = perturbation[exp2_mask]
    output.loc[exp2_mask, 'PoM'] = new_pom[exp2_mask]
    output.loc[exp2_mask, 'state'] = exp2_state_series[exp2_mask]

    print()
    print(output.loc[exp2_mask, 'state'].value_counts())
    print()
    print(output.loc[exp2_mask, ['Full_Path', 'PoM_old', 'PoM', 'delta', 'experiment_delta', 'state', 'previous_state']].head(10).to_string())

# Round delta/PoM to 6 decimals -- the convergence + perturbation additions
# above (and assign_deltas' reindex/fillna resolution before that) can leave
# harmless float noise like -1.3877787807814457e-17 instead of an exact 0.0.
# Doesn't change any actual value, just its representation.
output['delta'] = output['delta'].round(6)
output['PoM'] = output['PoM'].round(6)

# state_sequence: this segment's full distinct-state history, delimited with
# " | ". Carried forward from reference_df's own state_sequence, with the
# FINAL state (after both the main resolution above and the experiment 2
# overlay have had their say) appended ONLY when it's an actual transition
# (state != previous_state) -- so the sequence only ever records distinct
# states, not one entry per round; "no change" (state == previous_state,
# which also covers retained/holdout/pure-rollback) leaves it untouched.
prev_seq_by_key = dict(zip(zip(reference_df['Third_Party_Data_Provider_Id'], reference_df['Full_Path']),
                            reference_df['state_sequence']))
output['state_sequence'] = [prev_seq_by_key.get((p, f), 'O')
                             for p, f in zip(output['Third_Party_Data_Provider_Id'], output['Full_Path'])]
_transitioned = output['state'] != output['previous_state']
output.loc[_transitioned, 'state_sequence'] = (
    output.loc[_transitioned, 'state_sequence'] + ' | ' + output.loc[_transitioned, 'state']
)
print(f"\n{int(_transitioned.sum()):,} segment(s) transitioned to a new state this round")
print()
print(output['state'].value_counts())


364 / 364 experiment2 candidates found in output

state
C2           148
T2 +0.025    109
T2 -0.025    107
Name: count, dtype: int64

                                                                              Full_Path  PoM_old    PoM  delta  experiment_delta      state previous_state
2167                                  Asia > Arts & Entertainment > Humor > Live Comedy     0.15  0.100 -0.050             0.000         C2              O
2169                                               Asia > Arts & Entertainment > Movies     0.15  0.100 -0.050             0.000         C2              O
2190      Asia > Arts & Entertainment > Music & Audio > Experimental & Industrial Music     0.15  0.125 -0.025             0.025  T2 +0.025              O
2200                            Asia > Arts & Entertainment > Music & Audio > Pop Music     0.15  0.125 -0.025             0.025  T2 +0.025              O
2237                                   Asia > Autos & Vehicles > Bicycles & Accessories    

In [11]:
# per-rule hit counts -- sanity check that each rule matched what you expected.
# With a large reconstructed rule list (one exact-match rule per segment) or
# a rule with a large Full_Path list value, print a summary instead of
# dumping every rule/list inline.
def _rule_repr(rule, max_list_items=5):
    parts = []
    for k, v in rule.items():
        if isinstance(v, (list, tuple)) and len(v) > max_list_items:
            v = f"[{len(v)} items: {v[0]!r}, {v[1]!r}, ... ]"
        parts.append(f"{k}={v!r}")
    return "{" + ", ".join(parts) + "}"

if len(delta_rules) <= 20:
    for i, rule in enumerate(delta_rules):
        matched = apply_rule(segments, rule)
        print(f"rule {i}: {_rule_repr(rule)}")
        print(f"  -> {len(matched):,} segments matched")
else:
    hit_counts = [len(apply_rule(segments, rule)) for rule in delta_rules]
    print(f"{len(delta_rules):,} rules -- hit-count distribution:")
    print(pd.Series(hit_counts).value_counts().sort_index())
    n_zero_hit = sum(1 for c in hit_counts if c == 0)
    if n_zero_hit:
        print(f"WARNING: {n_zero_hit} rule(s) matched zero segments")

3,352 rules -- hit-count distribution:
1    3352
Name: count, dtype: int64


In [12]:
if output_file is not None:
    output.to_csv(output_file, index=False)
    print(f"wrote {len(output):,} rows to {output_file}")
else:
    print("output_file is None -- not writing to disk")

wrote 5,734 rows to /Users/ravirajan/code/outputs/TTD/Price_Assign_2026-08-03.csv
